# MCM LTER Aquarius Data Processing

This script was created by Jared Collins in the summer of 2025 to be used by the MCM LTER stream team as an easy way to merge and organize data downloaded from Aquarius. The original processing code was done in R. The hope is this python script will be easier to use and more intuitive, allowing users to easily convert data files into publishable forms. In the future, it may need to be edited/changed to reflect changes in data collection strategies.

## Importing Packages + Setting Directory

It is very important that the computer being used for processing contains the proper packages, paths, and names for the data files that will be used. **The script expects the data files from Aquarius to be stored in the downloads folder with no edits made to the name of the file. Please ensure that this is the case. This section only needs to be ran once per session.**

In [77]:
from datetime import date
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [78]:
home = Path.home()
downloads_path = home / "Downloads"
downloads_path = downloads_path.as_posix()
print("Downloads folder:", downloads_path)

Downloads folder: C:/Users/jared/Downloads


In [79]:
streamid_map = {
    'B3': 'lawson_b3',
    'B5': 'bohner_b5',
    'C1': 'common_c1',
    'F1': 'canada_f1',
    'F2': 'huey_f2',
    'F3': 'lostseal_f3',
    'F5': 'aiken_f5',
    'F6': 'vonguerard_f6',
    'F7': 'harnish_f7',
    'F8': 'crescent_f8',
    'F9': 'green_f9',
    'F10': 'delta_f10',
    'H1': 'andersen_h1',
    'M1': 'adams_m1',
    'M2': 'miers_m2',
    'Onyx_LWRT': 'onyx_lwrt',
    'OnyxVanda': 'onyx_vanda'}

The following cell contains a list for the script to detect which stream is being worked on and how to fill in the data appropriately. This should never have to be changed.

## Specifying Data (EDIT THIS SECTION ONLY)

In order for this script to work properly, you will need to input necessary information about the files that you are hoping to publish:
- Stream ID: Acceptable formats are B3, B5, C1, F1, F2, F3, F5, F6, F7, F8, F9, F10, H1, M1, M2, Onyx_LWRT, OnyxVanda
- Season: Separate seasons into A and B data depending on what portion of the data you are working on; will only impact how datafile is saved
- Date: The first day of the data file, written in "YYYYMMDD" format

**Edit the following cell with the information above. If done correctly, this is the only cell you will have to edit.**

In [278]:
streamid ='F7'
season = '2425B'
date = '20250124'

## Organizing Data and Publishing CSV File

The next cell deals with calling the data file moved into the downloads folder. So long as the file is in the downloads folder of your computer and you did not change the name since the download, this cell should work perfectly.

In [279]:
dis = pd.read_csv(f'{downloads_path}/Discharge.Metric@{streamid}.{date}.csv', skiprows=14)
temp = pd.read_csv(f'{downloads_path}/Water_Temp.Working@{streamid}.{date}.csv', skiprows=14)
spc = pd.read_csv(f'{downloads_path}/Sp_Cond.Working@{streamid}.{date}.csv', skiprows=14)
dosat = pd.read_csv(f'{downloads_path}/Dis_Oxygen_Sat.Working@{streamid}.{date}.csv', skiprows=14)
streamid_lc = streamid.lower()

def clean_sensor_failures(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bad_values = [7999, -7999, 9999, -9999, '7999', '-7999', '9999', '-9999', '-INF', 'INF','NAN','nan']
    df.replace(bad_values, np.nan, inplace=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].astype(float)
    return df

def prep_sensor(df, value_name):
    df = clean_sensor_failures(df)
    df['date_time'] = pd.to_datetime(df['Timestamp (UTC+13:00)'])
    df = df[['date_time', 'Value']]
    df = df.rename(columns={'Value': value_name})
    return df
    
dis = prep_sensor(dis, 'discharge_ls')
temp = prep_sensor(temp, 'watertemp_degc')
spc = prep_sensor(spc, 'spcond_uscm')
dosat = prep_sensor(dosat, 'do_pct')

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/jared/Downloads/Discharge.Metric@F7.20250124.csv'

The following cell deals with creating the dataframe that will contain all of the data specified and ready for publishing, as well as generating the CSV file output. **It will publish this file to your downloads folder, so be sure to move it to the appropriate Stream Team folder once done.**

In [277]:
subm_file = (dis
    .merge(temp, on='date_time', how='outer')
    .merge(spc, on='date_time', how='outer')
    .merge(dosat, on='date_time', how='outer')
            )

subm_file = subm_file.set_index('date_time')
full_time_index = pd.date_range(start=subm_file.index.min(), end=subm_file.index.max(), freq='15min')
subm_file = subm_file.reindex(full_time_index)

subm_file = subm_file.reset_index()
subm_file = subm_file.rename(columns={'index': 'date_time'})
streamid_lc = streamid.lower()
subm_file['dataset_code'] = f'strm_hydro_{streamid_lc}'
subm_file['strmgageid'] = streamid_map[streamid]

subm_file = subm_file[['dataset_code', 'strmgageid', 'date_time', 
                       'discharge_ls', 
                       'watertemp_degc', 
                       'spcond_uscm', 
                       'do_pct'
                      ]]

subm_file.to_csv(f'{downloads_path}/{streamid}_{season}_SUBM.csv', index=False, na_rep='')